# TrustMind AI — Experiment 1: LLM-Only Baseline

**MSc Artificial Intelligence Dissertation**

## Research question

> To what extent does Retrieval-Augmented Generation (RAG) improve the trustworthiness, reliability, and explainability of LLM-generated wellbeing assessments compared with standalone LLMs?

## Purpose of this notebook

This notebook implements the **LLM-only baseline** (no retrieval, no external knowledge base).

Each SWMH test post is classified by a standalone GPT model using a fixed prompt. Results are saved for later comparison against the **LLM+RAG** experiment.

| Condition | Retrieval | Role |
|-----------|-----------|------|
| **LLM-only (this notebook)** | None | Baseline |
| LLM + RAG (next experiment) | Curated wellbeing corpus | Treatment |

### Model choice (controlled variable)

**GPT-4.1** (`gpt-4.1`) was selected as the underlying language model due to its strong performance in instruction following, natural language understanding, and structured output generation. The **same model** will be used in both the standalone LLM and LLM+RAG pipelines so that any observed differences in performance can be attributed to the addition of retrieval augmentation rather than differences in the underlying language model.

**Out of scope here:** RAG, FAISS, BM25, hybrid retrieval, external knowledge sources.


## Section 1 — Configuration

Change these values to re-run the experiment under different settings. Keep `RANDOM_SEED` fixed for reproducibility.

`MODEL_NAME` defaults to **`gpt-4.1`** for both this baseline and the forthcoming RAG experiment (controlled comparison).


In [ ]:
# ============================================================
# CONFIGURATION — edit these values as needed
# ============================================================
# GPT-4.1: strong instruction following + structured JSON output.
# Use the SAME model in LLM-only and LLM+RAG experiments.
MODEL_NAME = "gpt-4.1"       # controlled variable across baseline vs RAG
SAMPLE_SIZE = 100            # number of test posts to evaluate
RANDOM_SEED = 42             # reproducible sampling
TEMPERATURE = 0.0            # low temperature for more stable classifications

# Optional pacing / robustness
SLEEP_BETWEEN_CALLS = 0.5    # seconds between API calls (rate-limit courtesy)
MAX_RETRIES = 5              # retries per post on transient API failures
PROGRESS_EVERY = 10          # print progress every N posts

# Set True to skip API calls and verify the rest of the pipeline with mocks
DRY_RUN = False

print("MODEL_NAME:", MODEL_NAME)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("RANDOM_SEED:", RANDOM_SEED)
print("TEMPERATURE:", TEMPERATURE)
print("DRY_RUN:", DRY_RUN)


## Section 2 — Imports and Paths

Helpers live in `research/llm_baseline.py` so the same parsing / metrics code can be reused by the future RAG experiment.


In [ ]:
from pathlib import Path
import json
import os
import sys
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Resolve project root whether the notebook is started from repo root or research/
ROOT = Path.cwd().resolve()
if ROOT.name == "research":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "research"))

from llm_baseline import (
    VALID_LABELS,
    build_prompt,
    compute_metrics,
    load_and_sample_test,
    parse_prediction,
    run_baseline_inference,
)

# Load OPENAI_API_KEY from research/.env if present
load_dotenv(ROOT / "research" / ".env")
load_dotenv(ROOT / ".env")

DATA_DIR = ROOT / "datasets" / "swmh"
TEST_CSV = DATA_DIR / "test.csv"
RESULTS_DIR = ROOT / "research" / "results"
FIGURES_DIR = ROOT / "research" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = RESULTS_DIR / "llm_baseline_predictions.csv"
METRICS_PATH = RESULTS_DIR / "llm_baseline_metrics.json"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Project root:", ROOT)
print("Test CSV:", TEST_CSV)
print("Results dir:", RESULTS_DIR)
print("API key present:", bool(os.getenv("OPENAI_API_KEY")))


## Section 3 — Load and Sample the SWMH Test Set

We load `datasets/swmh/test.csv`, normalise labels from `self.depression` → `depression` (etc.), then draw a reproducible random sample of `SAMPLE_SIZE` posts.


In [ ]:
if not TEST_CSV.exists():
    raise FileNotFoundError(
        f"Missing {TEST_CSV}. Place SWMH test.csv under datasets/swmh/."
    )

sample_df = load_and_sample_test(
    TEST_CSV,
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
)

print(f"Sampled {len(sample_df)} posts (requested SAMPLE_SIZE={SAMPLE_SIZE})")
print("\nTrue label distribution in sample:")
print(sample_df["true_label"].value_counts().reindex(VALID_LABELS).fillna(0).astype(int))
sample_df.head(3)


## Section 4 — Prompt Template (LLM-only)

The prompt asks for **exactly one** class label and a JSON object containing `predicted_label`, `confidence`, and `reasoning`. No documents are retrieved.


In [ ]:
demo_prompt = build_prompt("<reddit post>")
print(demo_prompt)


## Section 5 — Run LLM-Only Inference

Each post is sent **individually** to the configured OpenAI model.

Error handling (continues instead of crashing):
- API failures / timeouts
- Rate limiting (exponential backoff)
- Empty responses
- Invalid JSON / invalid labels

Set `OPENAI_API_KEY` in `research/.env` (see `research/.env.example`).

If `DRY_RUN=True`, mock predictions are generated so the evaluation / plotting cells still run offline.


In [ ]:
def run_dry_run(sample: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """Offline mock predictions for pipeline testing without API spend."""
    rng = np.random.default_rng(seed)
    rows = []
    labels = list(VALID_LABELS)
    for _, row in sample.iterrows():
        # Bias toward the true label so dry-run metrics are non-trivial but imperfect
        if rng.random() < 0.55:
            pred = row["true_label"]
        else:
            pred = labels[int(rng.integers(0, len(labels)))]
        rows.append({
            "text": row["text"],
            "true_label": row["true_label"],
            "predicted_label": pred,
            "confidence": float(rng.uniform(0.4, 0.95)),
            "reasoning": "DRY_RUN mock prediction — not a real LLM output.",
            "parse_ok": True,
            "error": "",
        })
    return pd.DataFrame(rows)


if DRY_RUN:
    print("DRY_RUN=True — generating mock predictions (no API calls).")
    predictions_df = run_dry_run(sample_df, seed=RANDOM_SEED)
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not found. Add it to research/.env or set DRY_RUN=True."
        )

    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    print(f"Running LLM-only inference with model={MODEL_NAME} on {len(sample_df)} posts...")
    predictions_df = run_baseline_inference(
        sample_df,
        client,
        model_name=MODEL_NAME,
        temperature=TEMPERATURE,
        sleep_between_calls=SLEEP_BETWEEN_CALLS,
        progress_every=PROGRESS_EVERY,
        max_retries=MAX_RETRIES,
    )

print("\nDone.")
print("Valid JSON predictions:", int(predictions_df["parse_ok"].sum()), "/", len(predictions_df))
print("Rows with errors:", int((predictions_df["error"].astype(str) != "").sum()))
predictions_df.head(3)


## Section 6 — Save Predictions

Persist the fields required for dissertation comparison:
`text`, `true_label`, `predicted_label`, `confidence`, `reasoning`.


In [ ]:
export_cols = ["text", "true_label", "predicted_label", "confidence", "reasoning"]
# Keep audit columns in CSV as well for debugging failed rows
save_df = predictions_df[export_cols + ["parse_ok", "error"]].copy()
save_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8")
print("Saved predictions →", PREDICTIONS_PATH)
print("Rows:", len(save_df))


## Section 7 — Evaluation Metrics

Reliability-oriented metrics for the baseline:
- Accuracy
- Precision / Recall / F1 (macro) — preferred under class imbalance (see EDA)
- Confusion matrix
- Full classification report

Invalid or empty predictions count as incorrect for accuracy.


In [ ]:
y_true = predictions_df["true_label"].tolist()
y_pred = predictions_df["predicted_label"].tolist()

metrics = compute_metrics(y_true, y_pred)

print("=" * 60)
print("LLM-ONLY BASELINE METRICS")
print("=" * 60)
print(f"Model:              {MODEL_NAME}")
print(f"Sample size:        {metrics['n_samples']}")
print(f"Valid predictions:  {metrics['n_valid_predictions']}")
print(f"Invalid predictions:{metrics['n_invalid_predictions']}")
print(f"Accuracy:           {metrics['accuracy']:.4f}")
print(f"Precision (macro):  {metrics['precision_macro']:.4f}")
print(f"Recall (macro):     {metrics['recall_macro']:.4f}")
print(f"F1-score (macro):   {metrics['f1_macro']:.4f}")
print("\nClassification report:")
print(metrics["classification_report_text"])
print("Confusion matrix (rows=true, cols=predicted):")
print("Labels:", metrics["labels"])
print(np.array(metrics["confusion_matrix"]))


In [ ]:
metrics_payload = {
    "experiment": "llm_only_baseline",
    "research_question": (
        "To what extent does Retrieval-Augmented Generation (RAG) improve the "
        "trustworthiness, reliability, and explainability of LLM-generated "
        "wellbeing assessments compared with standalone LLMs?"
    ),
    "model_name": MODEL_NAME,
    "sample_size": SAMPLE_SIZE,
    "random_seed": RANDOM_SEED,
    "temperature": TEMPERATURE,
    "dry_run": DRY_RUN,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "n_samples": metrics["n_samples"],
        "n_valid_predictions": metrics["n_valid_predictions"],
        "n_invalid_predictions": metrics["n_invalid_predictions"],
        "accuracy": metrics["accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
        "labels": metrics["labels"],
        "confusion_matrix": metrics["confusion_matrix"],
        "classification_report": metrics["classification_report"],
    },
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print("Saved metrics →", METRICS_PATH)


## Section 8 — Visualisations

Figures are saved under `research/figures/` for the dissertation write-up.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

labels = list(VALID_LABELS)
cm = np.array(metrics["confusion_matrix"])

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
ax.set_title(f"LLM-only baseline confusion matrix\n({MODEL_NAME}, n={metrics['n_samples']})")
fig.tight_layout()
cm_path = FIGURES_DIR / "llm_baseline_confusion_matrix.png"
fig.savefig(cm_path)
plt.show()
print("Saved:", cm_path)


In [ ]:
pred_counts = (
    predictions_df["predicted_label"]
    .value_counts()
    .reindex(labels, fill_value=0)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(pred_counts.index.astype(str), pred_counts.values, color="#2563eb", edgecolor="white")
ax.set_title("Predicted label distribution (LLM-only baseline)")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Count")
plt.xticks(rotation=30, ha="right")
fig.tight_layout()
pred_path = FIGURES_DIR / "llm_baseline_predicted_labels.png"
fig.savefig(pred_path)
plt.show()
print("Saved:", pred_path)


In [ ]:
true_counts = predictions_df["true_label"].value_counts().reindex(labels, fill_value=0)
pred_counts = predictions_df["predicted_label"].value_counts().reindex(labels, fill_value=0)

x = np.arange(len(labels))
width = 0.38

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width / 2, true_counts.values, width, label="True", color="#0d9488", edgecolor="white")
ax.bar(x + width / 2, pred_counts.values, width, label="Predicted", color="#2563eb", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Count")
ax.set_title("True vs predicted label counts (LLM-only baseline)")
ax.legend()
fig.tight_layout()
cmp_path = FIGURES_DIR / "llm_baseline_true_vs_predicted.png"
fig.savefig(cmp_path)
plt.show()
print("Saved:", cmp_path)


## Section 9 — Qualitative Spot-Check

Inspect a few correct and incorrect predictions. Reasoning strings support later **explainability** discussion (baseline has no retrieved sources to cite).


In [ ]:
predictions_df["correct"] = predictions_df["true_label"] == predictions_df["predicted_label"]

print("Accuracy on sample:", predictions_df["correct"].mean())
print("\n--- Example CORRECT predictions ---")
for _, row in predictions_df[predictions_df["correct"]].head(2).iterrows():
    print(f"True={row['true_label']} | Pred={row['predicted_label']} | conf={row['confidence']:.2f}")
    print("Text:", str(row["text"])[:240].replace("\n", " "), "...")
    print("Reasoning:", str(row["reasoning"])[:300])
    print()

print("--- Example INCORRECT predictions ---")
for _, row in predictions_df[~predictions_df["correct"]].head(2).iterrows():
    print(f"True={row['true_label']} | Pred={row['predicted_label']} | conf={row['confidence']:.2f}")
    print("Text:", str(row["text"])[:240].replace("\n", " "), "...")
    print("Reasoning:", str(row["reasoning"])[:300])
    print()


## Section 10 — Dissertation Summary (LLM-only Baseline)

### What this experiment measures

This notebook establishes how well a **standalone LLM (GPT-4.1)** can map informal Reddit wellbeing posts to SWMH subreddit labels using prompt-based classification only. GPT-4.1 is held constant in the RAG arm so performance gaps reflect retrieval, not model substitution.

### Link to the research question

| Construct | How the baseline contributes |
|-----------|------------------------------|
| **Reliability** | Macro-F1 / accuracy / confusion matrix on the same sampled test posts |
| **Trustworthiness** | Confidence values + failure/abstention-like empty parses for later calibration analysis |
| **Explainability** | Free-text `reasoning` without external citations (contrast with RAG source traces later) |

### Outputs for the RAG comparison

- `research/results/llm_baseline_predictions.csv`
- `research/results/llm_baseline_metrics.json`
- `research/figures/llm_baseline_*.png`

### Next experiment

Implement **LLM + RAG** with the same sample, model, temperature, and metrics so differences can be attributed to retrieval grounding rather than data or prompt drift.

---

*End of LLM-only baseline notebook.*
